# Lab 2.4 - Function Calling and a Simple Agent (Start)
**Day 2 - Session 4**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day2/lab2_4_agent/start/lab2_4_agent.ipynb)

**Goal:** Give a model tools it can call, then let a smolagents `CodeAgent`
chain those tools to answer a question. Watch the ReAct loop of thought,
action, and observation. The stretch goal rebuilds the same idea in LangGraph.

Look for the `# TODO` markers and fill them in. The notebook in the `solution/`
folder has the full answer if you get stuck.

This lab needs an LLM API key (set in Colab or your local `.env`). Runs on
CPU; no GPU needed.

In [ ]:
# Scoped install for this lab (safe to re-run; preinstalled on Colab).
%pip install -q smolagents langgraph litellm fastapi python-dotenv

## Setup: make the shared client available
This finds the `shared/` package locally, or clones the companion repo when
running on Colab. It also makes sure an API key is set. You do not need to
change this cell.

In [ ]:
import os
import subprocess
import sys


def _find_repo_root():
    d = os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(d, "shared")):
            return d
        d = os.path.dirname(d)
    return None


root = _find_repo_root()
if root is None:  # On Colab: clone the companion repo for the shared client.
    if not os.path.isdir("gen-ai-companion-intuite"):
        subprocess.run(
            ["git", "clone", "-q",
             "https://github.com/kpassoubady/gen-ai-companion-intuite.git"],
            check=True,
        )
    root = "gen-ai-companion-intuite"
sys.path.insert(0, root)

from shared import get_completion_full, get_model, show_config

if not any(os.environ.get(k) for k in ("OPENAI_API_KEY", "GEMINI_API_KEY",
                                       "ANTHROPIC_API_KEY", "AZURE_API_KEY")):
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        from getpass import getpass
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")
    os.environ.setdefault("LLM_PROVIDER", "openai")

show_config()

## 1. Define two tools
A tool is a plain function with a docstring and typed arguments. The `@tool`
decorator exposes it to the agent. The model reads the name and docstring to
decide when to call it. A clear docstring with an `Args:` section is required.

In [ ]:
from smolagents import tool


@tool
def get_tax_rate(state_code: str) -> float:
    """Return the sales tax rate as a percent for a US state.

    Args:
        state_code: Two-letter state code, for example CA or NY.
    """
    rates = {"CA": 7.25, "NY": 4.0, "TX": 6.25, "WA": 6.5, "FL": 6.0}
    return rates.get(state_code.upper(), 0.0)


# TODO: Write a second tool `calculate_total(amount: float, tax_rate_percent:
# float) -> float` that returns the amount plus tax, rounded to 2 decimals.
# Decorate it with @tool and give it a docstring with an Args: section.
# Hint: return round(amount * (1 + tax_rate_percent / 100.0), 2)


print("Sanity check:", calculate_total(250.0, get_tax_rate("CA")))

## 2. Build a CodeAgent
`LiteLLMModel` routes to whatever provider the shared config selects, so the
agent uses the same model as the other labs. We cap `max_steps` so the loop
cannot run away.

In [ ]:
from smolagents import CodeAgent, LiteLLMModel

model = LiteLLMModel(model_id=get_model("mini"))
# TODO: Create a CodeAgent with your two tools, the model, and max_steps=5.
# Hint: agent = CodeAgent(tools=[get_tax_rate, calculate_total], model=model,
#                         max_steps=5)
agent = ...

## 3. Run the agent
The agent reasons, calls `get_tax_rate`, then `calculate_total`, and returns
the answer. The printed log shows each thought, action, and observation.

In [ ]:
# TODO: Call agent.run(...) with a question that needs both tools, for example
# the total cost of a $250 purchase in California after sales tax.
answer = ...
print("\nFinal answer:", answer)

The total should be about 268.12 (250 plus 7.25% tax). The log above is the
ReAct loop in action: think, act with a tool, observe the result, repeat.

## Stretch goals
Try these if you finish early. Hints only here; the full code is in the
solution notebook.

1. **Add a third tool** (for example a currency converter) and ask a question
   that needs all three.
2. **Rebuild in LangGraph.** Make the loop explicit: an `agent` node calls the
   model with `get_completion_full(..., tools=TOOL_SCHEMAS)`, a `tools` node
   runs any requested tool, and a conditional edge loops until there are no
   more tool calls. See the solution for a full working graph.

In [ ]:
# TODO (stretch): your experiments here.